In [0]:
from pyspark.sql import functions as F

In [0]:
CATALOG = "airline_analytics"
HIST = f"{CATALOG}.bronze.flights_historical"
RECENT = f"{CATALOG}.bronze.flights_recent"

In [0]:
hist = spark.table(HIST)
recent = spark.table(RECENT)
 
print(f"{HIST}\n  rows: {hist.count():,}  cols: {len(hist.columns)}")
print(f"{RECENT}\n  rows: {recent.count():,}  cols: {len(recent.columns)}")

In [0]:
coverage = (
    recent.groupBy("Year")
    .agg(
        F.countDistinct("Month").alias("months_present"),
        F.count("*").alias("rows"),
        F.min("FlightDate").alias("min_date"),
        F.max("FlightDate").alias("max_date"),
    )
    .orderBy("Year")
)
display(coverage)

In [0]:
# Per-month row counts: a doubled month stands out immediately against its neighbours.
display(
    recent.groupBy("Year", "Month")
    .agg(F.count("*").alias("rows"))
    .orderBy("Year", "Month")
)

In [0]:
# Historical: per-year counts, sanity-checked against ~7M US domestic flights/year.
display(
    hist.groupBy("Year", "Month").agg(F.count("*").alias("rows")).orderBy("Year", "Month")
)

In [0]:
recent_key = [
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
]

total = recent.count()
distinct = recent.select(*recent_key).distinct().count()
print(f"rows: {total:,}")
print(f"distinct keys: {distinct:,}")
print(f"gap: {total - distinct:,}  ({(total - distinct) / total:.4%})")

In [0]:
if total != distinct:
    display(
        recent.groupBy(*recent_key)
        .agg(F.count("*").alias("n"))
        .filter(F.col("n") > 1)
        .groupBy(F.year("FlightDate").alias("Year"), F.month("FlightDate").alias("Month"))
        .agg(F.count("*").alias("dup_keys"), F.sum("n").alias("dup_rows"))
        .orderBy("Year", "Month")
    )

In [0]:
HIST_CORE = [
    "Year", "Month", "DayofMonth", "DayOfWeek",
    "UniqueCarrier", "TailNum", "FlightNum", "Origin", "Dest",
    "CRSDepTime", "DepTime", "DepDelay", "CRSArrTime", "ArrTime", "ArrDelay",
    "CRSElapsedTime", "ActualElapsedTime", "AirTime", "TaxiIn", "TaxiOut", "Distance",
    "Cancelled", "CancellationCode", "Diverted",
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay",
    "IsArrDelayed", "IsDepDelayed",
]
 
RECENT_CORE = [
    "Year", "Month", "DayofMonth", "DayOfWeek", "FlightDate",
    "Reporting_Airline", "Tail_Number", "Flight_Number_Reporting_Airline",
    "Origin", "OriginCityName", "OriginState", "Dest", "DestCityName", "DestState",
    "CRSDepTime", "DepTime", "DepDelay", "DepDelayMinutes", "DepDel15",
    "CRSArrTime", "ArrTime", "ArrDelay", "ArrDelayMinutes", "ArrDel15",
    "CRSElapsedTime", "ActualElapsedTime", "AirTime", "TaxiIn", "TaxiOut", "Distance",
    "Cancelled", "CancellationCode", "Diverted",
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay",
]

In [0]:
print("--- historical ---")
for c, t in hist.dtypes:
    if c in HIST_CORE:
        print(f"  {c:<36} {t}")

In [0]:
print("\n--- recent ---")
for c, t in recent.dtypes:
    if c in RECENT_CORE:
        print(f"  {c:<36} {t}")

In [0]:
print("\nmissing from historical:", [c for c in HIST_CORE if c not in hist.columns])
print("missing from recent:   ", [c for c in RECENT_CORE if c not in recent.columns])

In [0]:
for col in ["Cancelled", "CancellationCode", "Diverted", "IsArrDelayed", "IsDepDelayed"]:
    if col in hist.columns:
        vals = [r[0] for r in hist.select(col).distinct().limit(20).collect()]
        print(f"hist.{col:<18} -> {vals}")
 
print()
for col in ["Cancelled", "CancellationCode", "Diverted", "DepDel15", "ArrDel15"]:
    if col in recent.columns:
        vals = [r[0] for r in recent.select(col).distinct().limit(20).collect()]
        print(f"recent.{col:<18} -> {vals}")

In [0]:
def sentinel_report(df, cols, label):
    present = [c for c in cols if c in df.columns]
    exprs = []
    for c in present:
        s = F.col(c).cast("string")
        exprs.append(F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}__null"))
        exprs.append(F.sum(F.when(s.isin("NA", "", " "), 1).otherwise(0)).alias(f"{c}__na"))
    row = df.agg(*exprs).collect()[0].asDict()
    print(f"--- {label} (non-zero only) ---")
    for k, v in row.items():
        if v and v > 0:
            print(f"  {k:<44} {v:,}")
 
sentinel_report(hist, ["DepTime", "ArrTime", "ArrDelay", "DepDelay", "AirTime", "TailNum", "CancellationCode", "CarrierDelay"], "historical")
print()
sentinel_report(recent, ["DepTime", "ArrTime", "ArrDelay", "DepDelay", "AirTime", "Tail_Number", "CancellationCode", "CarrierDelay"], "recent")

In [0]:
for c in ["CRSDepTime", "DepTime", "CRSArrTime", "ArrTime"]:
    h = hist.filter(F.col(c).cast("int").isin(0, 2400)).count() if c in hist.columns else 0
    r = recent.filter(F.col(c).cast("int").isin(0, 2400)).count() if c in recent.columns else 0
    print(f"{c:<14} hist: {h:>10,}   recent: {r:>10,}")

In [0]:
hist_carriers = {r[0] for r in hist.select("UniqueCarrier").distinct().collect()}
recent_carriers = {r[0] for r in recent.select("Reporting_Airline").distinct().collect()}
 
print(f"historical carriers: {len(hist_carriers)}")
print(f"recent carriers:     {len(recent_carriers)}")
print(f"both:                {len(hist_carriers & recent_carriers)}")
print(f"historical only:     {sorted(hist_carriers - recent_carriers)}")
print(f"recent only:         {sorted(recent_carriers - hist_carriers)}")

In [0]:
hist_apts = {r[0] for r in hist.select("Origin").distinct().collect()}
recent_apts = {r[0] for r in recent.select("Origin").distinct().collect()}
 
print(f"historical airports: {len(hist_apts)}")
print(f"recent airports:     {len(recent_apts)}")
print(f"both:                {len(hist_apts & recent_apts)}")
print(f"historical only ({len(hist_apts - recent_apts)}): {sorted(hist_apts - recent_apts)[:40]}")
print(f"recent only ({len(recent_apts - hist_apts)}): {sorted(recent_apts - hist_apts)[:40]}")

In [0]:
shared_names = set(hist.columns) & set(recent.columns)
recent_only = [c for c in recent.columns if c not in shared_names and not c.startswith("_")]
 
print(f"identical column names across sources ({len(shared_names)}): {sorted(shared_names)}")
print(f"\nrecent-only columns ({len(recent_only)}):")
for i in range(0, len(recent_only), 4):
    print("   " + "  ".join(f"{c:<34}" for c in recent_only[i:i + 4]))